# deepface — Colab GPU 训练

在 Colab 免费版（T4 GPU）上训练 ResUNet 检测器。整个 notebook 从上到下依次运行即可。

**Runtime → Change runtime type → T4 GPU**（不选 GPU 也能跑，但那就失去意义了）。

断点保护：训练中每 500 步把 resume checkpoint 写到 Google Drive，会话断开后重跑训练单元格并把 `INIT` 指向 `.ckpt` 即可继续。

## 0. 一次性准备：把数据上传到 Google Drive

训练需要三份数据（都在本地仓库里，git-ignored，所以需要手动传一次）。在本地仓库根目录（Git Bash）打包：

```bash
tar -czf widerface.tar.gz -C data widerface      # WIDER train/val 图片 + wider_face_split 标注
tar -czf no_person.tar.gz -C data no_person      # COCO 筛出的无人场景图（负样本）
# crops.tar.gz 仓库里已有（LFS），不用重新打包
```

然后在 Google Drive 里建文件夹 **`MyDrive/deepface-data/`**，把这三个文件传上去：

- `crops.tar.gz`（约 110 MB，crop 数据集）
- `widerface.tar.gz`（约 1.7 GB）
- `no_person.tar.gz`

之后的每次 Colab 会话都会从 Drive 拷贝到本地盘使用，不用重传。

In [ ]:
#@title 1. 确认 GPU
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), '请先在 Runtime → Change runtime type 里选择 GPU'

In [ ]:
#@title 2. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DATA = '/content/drive/MyDrive/deepface-data'  #@param {type: "string"}
import os
assert os.path.isdir(DRIVE_DATA), f'{DRIVE_DATA} 不存在，请先完成第 0 步的数据上传'

In [ ]:
#@title 3. 克隆代码
REPO = 'https://github.com/JINXia-hz/deepface.git'  #@param {type: "string"}
# 如果仓库是 private，用 https://<token>@github.com/JINXia-hz/deepface.git
!git clone {REPO} /content/deepface
%cd /content/deepface

In [ ]:
#@title 4. 把数据从 Drive 拷到本地盘并解压
# 直接读 Drive 上的几万张小图会非常慢，必须先拷到 /content 本地 SSD
!cp {DRIVE_DATA}/crops.tar.gz .
!mkdir -p data
!cp {DRIVE_DATA}/widerface.tar.gz {DRIVE_DATA}/no_person.tar.gz data/
!tar -xzf crops.tar.gz
!tar -xzf data/widerface.tar.gz -C data
!tar -xzf data/no_person.tar.gz -C data
!echo train_faces=$(ls train_images/1 | wc -l) train_nonface=$(ls train_images/0 | wc -l)
!ls data/widerface data/no_person | head

In [ ]:
#@title 5. 安装依赖
# Colab 已自带 CUDA 版 torch；requirements.txt 里 torch 未锁版本，已满足则跳过
!pip install -q -r requirements.txt

## 6. 训练

脚本：`python -m tools.train_deep <steps> <lambda> <out> <seed> [init] [tiles] [lr] [device] [batch]`

- `OUT` 指向 Drive，所以每 500 步的 `.ckpt` 和最终 `.pth` 都会持久化；断线后把 `INIT` 设为 `OUT + '.ckpt'` 再跑一遍即可续训（`STEPS` 是**本次**要跑的步数，总步数记录在 ckpt 里）
- 建议先 `STEPS=2000` 复现 baseline（对照 README：CPU 上 2000 步约 2.8 h，T4 上预计几分钟），确认 valid center-acc 在 0.95+，再放大
- 放大时优先加步数（如 20000–50000）；`BATCH` 可以提到 `'32,12'` 左右提高 GPU 利用率
- 免费版会话上限约 12 h，长训练拆成几段跑即可（ckpt 接力）

In [ ]:
import subprocess

STEPS = 2000          #@param {type: "integer"}
LAMBDA = 1.0          # 场景损失权重
SEED = 0              #@param {type: "integer"}
TILES = '192'         # 训练 tile 尺寸，逗号分隔可交替
LR = 1e-3             #@param {type: "number"}
BATCH = '10,4'        # 每步场景 tile 数, 负样本 tile 数
OUT = DRIVE_DATA + '/model_deep_colab.pth'
INIT = ''             # 续训时填 OUT + '.ckpt'

args = ['python', '-m', 'tools.train_deep', str(STEPS), str(LAMBDA), OUT, str(SEED)]
if INIT:
    args.append(INIT)
args += [TILES, str(LR), 'cuda', BATCH]
print(' '.join(args))
subprocess.run(args, check=True)

## 7. 完成后

1. 从 Drive 的 `deepface-data/` 下载 `model_deep_colab.pth` 到本地 `models/`
2. 本地验证：`py -m tools.detect samples/test.jpg models/model_deep_colab.pth`（`.pth` 保存时已转成 CPU 格式，无需 GPU）
3. 全面评估（`tools.eval_fcn` / `tools.eval_wider`）在本地 CPU 跑即可，流程与现有模型完全一致

如果新模型更好，把它提升为默认：改名放到 `models/model_deep.pth`（或改 `deepface/detect.py` 的默认路径），并按 README 的惯例把指标记进 README。